<div style="border: 3px solid #b42318; background: #fef3f2; color: #7a271a; border-radius: 12px; padding: 16px 20px; line-height: 1.5">
<div style="font-size: 20px; font-weight: 800; color: #b42318; margin-bottom: 10px">
&#9888;&#65039; ЧЕРНОВИК &mdash; НЕ АКТУАЛЬНАЯ ВЕРСИЯ
</div>
<p style="margin: 0 0 10px 0"><b>Это занятие ещё в работе и будет переписано.</b>
Формулировки, данные и порядок заданий изменятся; часть материала может
опираться на то, что к этому моменту курса ещё не прочитано.</p>
<p style="margin: 0">Заниматься по нему пока не нужно &mdash; дождитесь
окончательной версии. Актуально сейчас только <b>занятие&nbsp;1</b>.</p>
</div>

# Лабораторная работа 7. Метрические и байесовские методы классификации

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| Место в курсе | после лекции 6 |
| Опора на лекции | лекция 6: обобщённый метрический классификатор (опр. 6.1), выбор параметров скользящим контролем (§6.2), отступ (опр. 6.4) и STOLP (зам. 6.6), байесовское решающее правило (теорема 6.8), наивный байесовский классификатор (опр. 6.10), нормальный дискриминантный анализ и LDA (теорема 6.16) |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Понять, что метрические методы опираются на расстояние — и потому полностью зависят от того, в каких единицах измерены признаки и сколько их. Сравнить наши модели с **байесовским оптимумом** там, где истинные плотности известны, и увидеть, чего стоит наивное предположение о независимости признаков.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab07_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Данные занятия — учебные, одинаковые у всех.** Числа на экране у преподавателя
> и у вас совпадают, поэтому можно сверяться с соседом и спорить о результате вслух.
> Индивидуальная таблица, порождённая по вашему ФИО, появится в домашней работе.
>
> Почти все учебные выборки курса берутся из `sklearn.datasets` — это либо
> готовые наборы, либо порождённые генератором:
> [7.1. Игрушечные наборы](https://scikit-learn.ru/stable/datasets/toy_dataset.html) ·
> [7.2. Реальные наборы](https://scikit-learn.ru/stable/datasets/real_world.html) ·
> [7.3. Генераторы выборок](https://scikit-learn.ru/stable/datasets/sample_generators.html)

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.datasets import load_wine
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

---
# Часть 1. Метрический метод целиком зависит от расстояния

Определение 6.1: обучающая выборка упорядочивается по расстоянию до объекта $x$,
и ответ определяется взвешенным голосованием соседей:

$$
a(x) = \arg\max_k \Gamma_k(x), \qquad
\Gamma_k(x) = \sum_{i=1}^{\ell}\bigl[y_{(i)} = k\bigr]\,w(i, x).
$$

Выбор весовой функции $w$ порождает всё семейство: $k$ ближайших соседей
($w = [i\le k]$), парзеновские окна, метод потенциальных функций. Реализуем их
в домашней работе, а сейчас — о том, от чего результат зависит сильнее всего.

Расстояние **не инвариантно к единицам измерения**: признак в рублях подавит
признак в долях единицы просто потому, что его числа больше. Это не тонкость
реализации, а свойство самого метода.

> **Напоминание — осторожно: «ядро» здесь другое.** В занятии 4 ядром называлась функция $K(x,x') = \langle\varphi(x),\varphi(x')\rangle$,
> подходящая под критерий Мерсера, — способ работать в спрямляющем пространстве.
>
> Здесь **ядро сглаживания** — это просто невозрастающая функция $K(r)$ одного
> аргумента $r = \rho(x, x_i)/h$, задающая, с каким весом учитывается сосед на
> относительном расстоянии $r$: например, прямоугольное $K(r) = [r\le1]$,
> треугольное $K(r) = (1-r)_+$, квартическое $K(r)=(1-r^2)^2_+$,
> гауссовское $K(r) = e^{-r^2/2}$. Никакого скалярного произведения и никакого
> критерия Мерсера от неё не требуется.
>
> Слово одно, смыслы разные — так исторически сложилось. Понять, о каком идёт
> речь, легко по числу аргументов: у ядра Мерсера их два, у ядра сглаживания один.

In [ ]:
X_w, y_w = load_wine(return_X_y=True)
Xa, Xv, ya, yv = train_test_split(X_w, y_w, test_size=0.35, stratify=y_w,
                                  random_state=RANDOM_STATE)
sc = StandardScaler().fit(Xa)

raw = KNeighborsClassifier(5).fit(Xa, ya).score(Xv, yv)
scaled = KNeighborsClassifier(5).fit(sc.transform(Xa), ya).score(sc.transform(Xv), yv)
print(f"wine, kNN(5) без масштабирования : точность {raw:.4f}")
print(f"wine, kNN(5) со StandardScaler   : точность {scaled:.4f}")

spread = pd.Series(X_w.std(axis=0), index=load_wine().feature_names).sort_values()
print(f"\nразброс масштабов: от {spread.iloc[0]:.3f} ({spread.index[0]}) "
      f"до {spread.iloc[-1]:.1f} ({spread.index[-1]}) -- отношение {spread.iloc[-1] / spread.iloc[0]:.0f}")

### Задание 1.1. Проклятие размерности

Вторая беда — размерность. В $\mathbb{R}^n$ при больших $n$ все попарные
расстояния становятся почти одинаковыми, и понятие «ближайший сосед» теряет
смысл. Измерьте относительный контраст
$(\max\rho - \min\rho)/\overline{\rho}$ как функцию $n$.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def pairwise_dist(P):
    """Попарные расстояния (тождество из занятия 1)."""
    raise NotImplementedError


gen = np.random.default_rng(RANDOM_STATE)
rows = []
for n_dim in [1, 2, 3, 5, 10, 20, 50, 100, 300, 1000]:
    # TODO (2-3 строки): 500 точек из U[0,1]^n, попарные расстояния (без диагонали),
    #                    относительный контраст (max - min) / mean
    rows.append({"размерность": n_dim, "контраст": ...})
cd = pd.DataFrame(rows).set_index("размерность")
display(cd.round(3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogx(cd.index, cd["контраст"], "o-", lw=2, color="#128C7E")
ax1.set_xlabel("размерность $n$"); ax1.set_ylabel(r"$(\max\rho - \min\rho)/\overline{\rho}$")
ax1.set_title("Контраст расстояний исчезает с ростом размерности")

for n_dim in [2, 10, 100, 1000]:
    D = pairwise_dist(gen.uniform(0, 1, size=(400, n_dim)))
    off = D[~np.eye(400, dtype=bool)]
    ax2.hist(off / off.mean(), bins=60, histtype="step", lw=2, density=True,
             label=f"$n = {n_dim}$")
ax2.set_xlabel(r"$\rho / \overline{\rho}$"); ax2.set_ylabel("плотность")
ax2.set_title("Все расстояния сходятся к среднему"); ax2.legend()
plt.tight_layout(); plt.show()

> **Вывод.** Во сколько раз изменилась точность после масштабирования? Как ведёт себя контраст с ростом размерности и почему это убивает метрические методы?
>
> *(ваш ответ здесь)*

---
# Часть 2. Выбор числа соседей

Параметр $k$ — это компромисс смещение–разброс в чистом виде: при $k=1$
классификатор идеально описывает обучающую выборку и сильно от неё зависит,
при больших $k$ граница сглаживается вплоть до константы.

In [ ]:
Xa_s, Xv_s = sc.transform(Xa), sc.transform(Xv)
ks = list(range(1, 41))
cv7 = StratifiedKFold(5, shuffle=True, random_state=0)

cv_scores = [cross_val_score(KNeighborsClassifier(k), Xa_s, ya, cv=cv7).mean() for k in ks]
train_scores = [KNeighborsClassifier(k).fit(Xa_s, ya).score(Xa_s, ya) for k in ks]
k_best = ks[int(np.argmax(cv_scores))]

fig, ax = plt.subplots()
ax.plot(ks, train_scores, "s-", lw=1.5, ms=4, label="на обучающей выборке")
ax.plot(ks, cv_scores, "o-", lw=2, label="скользящий контроль")
ax.axvline(k_best, ls="--", color="#C97A2B", label=f"$k^* = {k_best}$")
ax.set_xlabel("число соседей $k$"); ax.set_ylabel("точность"); ax.legend()
ax.set_title("Выбор $k$: сложность модели убывает слева направо")
plt.tight_layout(); plt.show()
print(f"k = 1:  обучение {train_scores[0]:.3f}, контроль {cv_scores[0]:.3f}")
print(f"k = {k_best}: обучение {train_scores[k_best - 1]:.3f}, контроль {cv_scores[k_best - 1]:.3f}")

> **Вывод.** Чему равна точность на обучающей выборке при $k=1$ и почему? В какую сторону растёт сложность модели по оси $k$?
>
> *(ваш ответ здесь)*

---
# Часть 3. Байесовский оптимум: сколько вообще можно выжать

Теорема 6.8: при известных априорных вероятностях $P_k$ и плотностях $p_k(x)$
минимум среднего риска даёт правило $a(x) = \arg\max_k P_k\,p_k(x)$.
Это **оптимальный** классификатор: ни один алгоритм не может ошибаться реже.

Обычно $p_k$ неизвестны. Но если мы **сами породили данные**, оптимум вычислим
точно — и появляется редкая возможность узнать, сколько ещё можно выиграть.

> **Напоминание — LDA и QDA.** Два способа применить правило $a(x) = \arg\max_k P_k\,p_k(x)$, оценив
> плотности как гауссовские, $p_k = \mathcal N(\mu_k, \Sigma_k)$.
>
> **QDA** (quadratic discriminant analysis) оценивает свою ковариационную матрицу
> для каждого класса. Логарифм плотности квадратичен по $x$, разность двух
> логарифмов тоже — отсюда «quadratic»: граница получается кривой второго порядка.
> Параметров много: по $n(n+1)/2$ на класс.
>
> **LDA** (linear discriminant analysis) предполагает, что ковариация у всех
> классов **общая**, $\Sigma_k \equiv \Sigma$. Тогда квадратичные члены в
> разности сокращаются, и граница становится **линейной** — отсюда название.
> Параметров втрое-вчетверо меньше, и на малых выборках LDA часто выигрывает у
> QDA даже когда его предположение неверно: экономия на дисперсии оценки
> перевешивает смещение. Тот же компромисс смещение–разброс, что и всегда.

In [ ]:
from scipy.stats import multivariate_normal

MU = [np.array([0.0, 0.0]), np.array([2.0, 1.4])]
COV = [np.array([[1.0, 0.75], [0.75, 1.0]]),        # РАЗНЫЕ ковариации
       np.array([[1.6, -0.9], [-0.9, 1.0]])]
PRIOR = [0.6, 0.4]


def sample_mixture(n, g):
    z = g.random(n) < PRIOR[1]
    X = np.where(z[:, None], g.multivariate_normal(MU[1], COV[1], n),
                 g.multivariate_normal(MU[0], COV[0], n))
    return X, z.astype(int)


def bayes_rule(X):
    """Оптимальное правило теоремы 6.8 при ИЗВЕСТНЫХ плотностях."""
    d0 = PRIOR[0] * multivariate_normal(MU[0], COV[0]).pdf(X)
    d1 = PRIOR[1] * multivariate_normal(MU[1], COV[1]).pdf(X)
    return (d1 > d0).astype(int)


g = np.random.default_rng(RANDOM_STATE)
X_tr, y_tr = sample_mixture(400, g)
X_te, y_te = sample_mixture(20_000, g)
bayes_acc = float(np.mean(bayes_rule(X_te) == y_te))
print(f"байесовский оптимум: точность {bayes_acc:.4f}")
print(f"неустранимая ошибка: {1 - bayes_acc:.4f} -- ниже неё не опустится никто")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

models = {"наивный Байес (Gaussian)": GaussianNB(),
          "LDA": LinearDiscriminantAnalysis(),
          "QDA": QuadraticDiscriminantAnalysis(),
          "kNN (k=15)": KNeighborsClassifier(15),
          "логистическая регрессия": LogisticRegression(max_iter=2000)}

# TODO (2-3 строки): обучите каждую модель на (X_tr, y_tr), посчитайте точность
#   на большой выборке (X_te, y_te) и зазор до байесовского оптимума.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-4, 6, 300), np.linspace(-4, 5, 300))
grid = np.c_[g1.ravel(), g2.ravel()]
panels = [("байесовское правило", bayes_rule),
          ("наивный Байес", GaussianNB().fit(X_tr, y_tr).predict),
          ("LDA", LinearDiscriminantAnalysis().fit(X_tr, y_tr).predict),
          ("QDA", QuadraticDiscriminantAnalysis().fit(X_tr, y_tr).predict)]

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, (nm, fn) in zip(axes, panels):
    ax.contourf(g1, g2, fn(grid).reshape(g1.shape), levels=[-.5, .5, 1.5],
                colors=["#cfe8e4", "#f6ddc4"])
    ax.contour(g1, g2, bayes_rule(grid).reshape(g1.shape), levels=[0.5],
               colors="black", linewidths=2, linestyles="--")
    ax.scatter(*X_tr[y_tr == 0].T, s=8, marker="o")
    ax.scatter(*X_tr[y_tr == 1].T, s=8, marker="s")
    ax.set_title(nm, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
axes[0].set_ylabel("пунктир — байесовская граница")
plt.tight_layout(); plt.show()

> **Вывод.** Кто ближе всех к оптимуму и почему? Почему LDA даёт прямую, а QDA — кривую? Что означает наличие неустранимой ошибки?
>
> *(ваш ответ здесь)*

---
# Часть 4. Цена наивного предположения

Определение 6.10: наивный байесовский классификатор считает признаки
независимыми внутри класса, $p_k(x) = \prod_j p_{kj}(x_j)$, откуда

$$
a(x) = \arg\max_k\Bigl(\ln P_k + \sum_j \ln p_{kj}(x_j)\Bigr).
$$

Предположение почти всегда ложно — измерим, во что это обходится.

Важная деталь эксперимента: разность средних не должна быть направлена вдоль
собственного вектора ковариационной матрицы. Иначе оптимальное направление
$\Sigma^{-1}(\mu_1-\mu_0)$ не зависит от $\rho$, наивное предположение ничего
не портит, и эффекта просто не будет видно.

In [ ]:
MU1 = np.array([1.8, 0.0])
rows = []
for corr in [0.0, 0.3, 0.6, 0.85, 0.95, 0.99]:
    C = np.array([[1.0, corr], [corr, 1.0]])
    gg = np.random.default_rng(RANDOM_STATE)
    z = gg.random(600) < 0.5
    Xc = np.where(z[:, None], gg.multivariate_normal(MU1, C, 600),
                  gg.multivariate_normal([0, 0], C, 600))
    yc = z.astype(int)
    Xa2, Xv2, ya2, yv2 = train_test_split(Xc, yc, test_size=0.4, random_state=0, stratify=yc)
    rows.append({"корреляция признаков": corr,
                 "наивный Байес": GaussianNB().fit(Xa2, ya2).score(Xv2, yv2),
                 "LDA (учитывает ковариацию)": LinearDiscriminantAnalysis()
                 .fit(Xa2, ya2).score(Xv2, yv2)})
nb_tab = pd.DataFrame(rows).set_index("корреляция признаков")
display(nb_tab.round(4))

In [ ]:
fig, ax = plt.subplots()
nb_tab.plot(marker="o", lw=2, ax=ax)
ax.set_xlabel("корреляция признаков внутри класса"); ax.set_ylabel("точность")
ax.set_title("Цена наивного предположения о независимости")
plt.tight_layout(); plt.show()

> **Вывод.** При какой корреляции наивный Байес начинает заметно проигрывать? Почему на текстах он при этом работает хорошо, хотя слова заведомо зависимы?
>
> *(ваш ответ здесь)*

---
# Часть 5. Пять методов на одной задаче

Вернёмся к `wine`, с которого начали часть 1: 178 объектов, 13 признаков, три
сорта вина. Подберём $k$ скользящим контролем и сравним все методы занятия —
и заодно измерим, сколько стоит забытое масштабирование.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Xtr, Xte, ytr, yte = train_test_split(
    X_w, y_w, test_size=0.25, random_state=RANDOM_STATE, stratify=y_w)

# Масштабирование внутри Pipeline: на каждом фолде оно считается заново
# по обучающей части фолда (занятие 5, часть 3).
knn_pipe = lambda k: Pipeline([("scale", StandardScaler()),
                               ("knn", KNeighborsClassifier(k))])
cv_own = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
ks_own = [1, 3, 7, 15, 31]
scores_own = [cross_val_score(knn_pipe(k), Xtr, ytr, cv=cv_own,
                              scoring="accuracy").mean() for k in ks_own]
k_own = ks_own[int(np.argmax(scores_own))]
display(pd.DataFrame({"k": ks_own, "CV accuracy": np.round(scores_own, 4)}).set_index("k"))
print(f"лучшее k по скользящему контролю: {k_own}")

In [ ]:
rows = []
for name, m in [(f"kNN (k={k_own}), со шкалированием", knn_pipe(k_own)),
                (f"kNN (k={k_own}), БЕЗ шкалирования", KNeighborsClassifier(k_own)),
                ("наивный Байес", GaussianNB()),
                ("LDA", LinearDiscriminantAnalysis()),
                ("QDA", QuadraticDiscriminantAnalysis(reg_param=0.05)),
                ("логистическая регрессия",
                 Pipeline([("scale", StandardScaler()),
                           ("lr", LogisticRegression(max_iter=5000))]))]:
    m.fit(Xtr, ytr)
    rows.append({"модель": name, "accuracy на контроле": m.score(Xte, yte)})
vals, cnts = np.unique(ytr, return_counts=True)
rows.append({"модель": "константа (частый класс)",
             "accuracy на контроле": float((yte == vals[cnts.argmax()]).mean())})
display(pd.DataFrame(rows).set_index("модель").round(4))

> **Вывод.** Сколько стоило забытое масштабирование? Почему наивный Байес не уступил более сложным методам, хотя его предположение здесь заведомо неверно?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Почему kNN обязательно требует масштабирования признаков, а решающее дерево — нет? Что в устройстве методов даёт это различие?
2. Байесовский оптимум на вашей задаче даёт точность 0.87, ваша модель — 0.86. Стоит ли продолжать улучшать модель?
3. Наивный Байес предполагает независимость признаков. Приведите ситуацию, где предположение полностью ложно, но метод всё равно даёт верный ответ.
4. Чем отличаются LDA и QDA по числу оцениваемых параметров? При каком размере выборки вы предпочтёте LDA, зная, что ковариации классов различны?

---

**Дома:** откройте `lab07_homework.ipynb` — там две задачи: обобщённый метрический классификатор (все пять методов из одной формулы) и отбор эталонов алгоритмом STOLP.